# E7: MG-детектор упрощения на ResNet-18 / CIFAR-10

Протокол — `PROTOCOL.md` (записан до запуска). Ноутбук возобновляемый: уже посчитанные запуски пропускаются, результаты пишутся на Google Drive.

**Runtime → Change runtime type → GPU (T4 достаточно).**

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Бандл и Drive
Загрузите `colab_resnet.zip` в панель Files слева (или положите на Drive и поправьте путь).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -oq /content/colab_resnet.zip -d /content/ 2>/dev/null || unzip -oq /content/drive/MyDrive/colab_resnet.zip -d /content/
%cd /content/colab_resnet
OUT = '/content/drive/MyDrive/e7_results'
!mkdir -p $OUT
# CIFAR-10 заранее, с видимым прогрессом (в первый раз Colab молча завис на скачивании)
!mkdir -p /content/data
!test -d /content/data/cifar-10-batches-py || (wget -c --show-progress -q https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz -P /content/data && tar -xzf /content/data/cifar-10-python.tar.gz -C /content/data)
!ls /content/data

## 2. Обучение с нуля (scratch)
Порядок рукавов — по важности: если сессия оборвётся, главное уже будет посчитано. Один запуск ≈ 6–10 мин на T4.

In [ ]:
!python -u run_resnet.py --scenario scratch --arms base lr100 freeze_head prune80 batch_up lr10 prune95 --seeds 20 21 22 --data /content/data --out $OUT --verbose

## 3. Дообучение предобученной ResNet-18 (finetune)

In [ ]:
!python -u run_resnet.py --scenario finetune --arms base freeze_head lr100 prune80 batch_up lr10 prune95 --seeds 20 21 22 --data /content/data --out $OUT --verbose

## 4. Анализ
MG и правило детектора заморожены (`detector_rule.json`), на E7 ничего не подбирается.

In [ ]:
!python analyse_resnet.py --root $OUT/scratch
!python analyse_resnet.py --root $OUT/finetune

In [ ]:
!cd $OUT/.. && zip -qr e7_results.zip e7_results -x '*.npz'
from google.colab import files
files.download('/content/drive/MyDrive/e7_results.zip')

Архив без `.npz` содержит таблицы. Сами логи (`logs_*.npz`, ~40 МБ на запуск из-за скетча обновлений) остаются на Drive — их тоже стоит забрать для локального повторного анализа.